# MaskInterpreter — PyTorch example

This notebook walks through the **PyTorch** `mask_interpreter` package end to end:

1. Build (or load) a frozen image-to-image **predictor** — an in-silico labeling model.
2. Wrap it in an `Image2ImageInterpreter` and train a **mask generator** that finds the
   input regions the predictor relies on.
3. Run the FOV **threshold sweep** (`Analyzer`) to score mask efficacy and write
   per-image tiffs.
4. **Visualize** the input, prediction, importance mask and the noise-adapted prediction.

It is fully self-contained: a tiny **synthetic** FOV dataset stands in for real data and a
toy region-reading predictor stands in for a pretrained in-silico UNet, so every cell runs
with no downloads or weights. Markers below show exactly where to swap in your own data and
model.

In [ ]:
import os
import tempfile

import numpy as np
import tifffile
import torch
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from mask_interpreter import Image2ImageInterpreter, freeze
from mask_interpreter.analyze import Analyzer
from mask_interpreter.train import Trainer
from mask_interpreter.config import LossConfig
from mask_interpreter.data.fov import FOVPatchDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("torch", torch.__version__, "| device:", DEVICE)

## 1. Data

Both `FOVPatchDataset` (training) and `Analyzer` (evaluation) accept FOVs **directly in
memory** — no tiff/CSV round-trip. Each FOV is a `dict` mapping role names to volumes; only
`input` and `target` are required, and `structure_seg` (0/255) enables the organelle-context
metric. Volumes are `(Z, Y, X)` (`(Y, X)` for 2D):

| role | key | normalized |
|---|---|---|
| label-free input | `input` | yes |
| fluorescence target | `target` | yes |
| organelle segmentation | `structure_seg` | no (0/255) |
| nucleus / membrane (optional) | `channel_dna` / `channel_membrane` | yes |

Below we synthesize a tiny dataset with a central "organelle" block. **To use real data**,
build `FOVS` from your own arrays — or pass a CSV/tiff list instead via
`FOVPatchDataset(image_list_csv=...)` / `Analyzer(data=...)`.

In [ ]:
WORK = tempfile.mkdtemp(prefix="mask_interpreter_example_")   # only for saved output tiffs
Z, Y, X = 8, 32, 32                      # FOV size
N_FOV = 3

rng = np.random.default_rng(0)
region = np.zeros((Z, Y, X), np.float32)
region[:, Y // 4:Y - Y // 4, X // 4:X - X // 4] = 1.0   # central "organelle"

FOVS = []                                # in-memory FOVs: one role->volume dict each
for i in range(N_FOV):
    signal = rng.standard_normal((Z, Y, X)).astype(np.float32)
    FOVS.append({
        "input": signal,                 # label-free input
        "target": signal * region,       # fluorescence ~ signal within the organelle
        "structure_seg": region * 255.0, # organelle segmentation (0/255) -> context metric
    })

print(f"built {len(FOVS)} in-memory FOVs | channels: {list(FOVS[0])}")

## 2. The frozen predictor

MaskInterpreter is **model-agnostic**: it wraps *any* frozen `nn.Module` mapping
`x -> prediction`. Here a toy predictor reads only the central block of each patch, standing
in for a pretrained in-silico labeling UNet.

**To use your own model**, replace `RegionImagePredictor(...)` with
`freeze(your_pretrained_unet)` — e.g. `mask_interpreter.unet.UNet(..., final_activation="linear")`
loaded from a checkpoint. `freeze` puts it in eval mode and disables its parameter
gradients; gradients still flow *through* it to the mask.

In [ ]:
def center_region(spatial):
    r = torch.zeros(1, 1, *spatial)
    sl = [slice(None), slice(None)] + [slice(s // 4, s - s // 4) for s in spatial]
    r[tuple(sl)] = 1.0
    return r


class RegionImagePredictor(nn.Module):
    """Toy image->image predictor that reads only a central block (stand-in for a
    pretrained in-silico UNet)."""

    def __init__(self, spatial):
        super().__init__()
        self.register_buffer("region", center_region(spatial))

    def forward(self, x):
        return x * self.region


PATCH = (8, 16, 16)                       # (Z, Y, X) patch the interpreter operates on
predictor = freeze(RegionImagePredictor(PATCH)).to(DEVICE)

interp = Image2ImageInterpreter(
    predictor, spatial_size=PATCH, ndim=3, in_channels=1, pred_channels=1,
    loss=LossConfig(pcc_target=0.9, noise_scale=1.5),
).to(DEVICE)

n_train = sum(p.numel() for p in interp.parameters() if p.requires_grad)
print(f"interpreter built | trainable params (mask generator): {n_train:,}")
print("predictor frozen:", all(not p.requires_grad for p in interp.predictor.parameters()))

## 3. Train the mask generator

`FOVPatchDataset` samples random patches from the FOVs; each batch is `(input, target)` but
the training is **self-supervised** — the loss only compares the predictor's output on the
original vs. the noise-adapted input, so the target channel is unused here (it feeds the
weighted-PCC variant). `Trainer` runs a plain PyTorch loop.

In [ ]:
ds = FOVPatchDataset(images=FOVS, input_col="input", target_col="target",
                     patch_size=PATCH, patches_per_image=8, norm=True, seed=0)
loader = DataLoader(ds, batch_size=4, shuffle=True)

trainer = Trainer(interp, lr=2e-3, device=DEVICE,
                  checkpoint_path=os.path.join(WORK, "mask_generator.pt"),
                  monitor="stop", term=None,          # no val split in this tiny demo
                  early_stop_monitor=None)
history = trainer.fit(loader, epochs=20)

last = history[-1]
print(f"epoch {last['epoch']}: pcc={last['pcc']:.3f}  "
      f"mask_size(mean)={last['importance_mask_size']:.3f}  total={last['total_loss']:.3f}")

## 4. Evaluate — FOV threshold sweep

`Analyzer` tiles each FOV into patches, runs the predictor and the interpreter, blends the
patches back with triangular overlap weighting, and reports the Pearson correlation between
the predictor's output on the original vs. the mask-induced noisy input (**mask efficacy**).

- `calc_unet_pcc` — the predictor's own quality (assembled prediction vs. target).
- `analyze_th(mode="regular", manual_th="full", save_image=True)` — writes the continuous
  mask and noisy prediction tiffs and the `*_results.csv` files.

In [ ]:
az = Analyzer(interp, images=FOVS, input_col="input", target_col="target",
              patch_size=(*PATCH, 1), xy_step=16, z_step=8,
              batch_size=4, device=DEVICE)

OUT = os.path.join(WORK, "predictions")

unet_pcc = az.calc_unet_pcc(OUT, images=range(N_FOV))
print("predictor PCC vs target (per FOV):")
print(unet_pcc)

pcc_df, mask_df, ctx_df = az.analyze_th(OUT, mode="regular", manual_th="full",
                                        images=range(N_FOV), save_image=True, seed=0)
print("\nmask efficacy (PCC, full mask):")
print(pcc_df)
print("\nmask size (fraction of voxels):")
print(mask_df)

## 5. Visualize

Load the per-image tiffs the sweep saved and show, for one z-slice: the input, target and
predictor output; the importance mask and its overlays; the noise-adapted input and
prediction. Files live under `predictions/<i>/` (globals) and `predictions/<i>/full/`
(threshold `"full"`).

In [ ]:
def auto_balance(img):
    img = img.astype(np.float32)
    lo, hi = np.percentile(img, (0.1, 99.9))
    return np.clip((img - lo) / (hi - lo + 1e-8), 0, 1)


def _read(path):
    if not os.path.exists(path):
        return None
    a = np.asarray(tifffile.imread(path)).astype(np.float32)
    return a[..., 0] if a.ndim == 4 else a          # drop trailing channel -> (Z, Y, X)


def visualize_results(out_dir, image_index=0, z_slice=None):
    base = os.path.join(out_dir, str(image_index))
    full = os.path.join(base, "full")
    imgs = {
        "Input": _read(os.path.join(base, f"input_{image_index}.tiff")),
        "Target": _read(os.path.join(base, f"target_{image_index}.tiff")),
        "Prediction": _read(os.path.join(base, f"unet_prediction_{image_index}.tiff")),
        "Importance mask": _read(os.path.join(full, f"mask_{image_index}.tiff")),
        "Noisy input": _read(os.path.join(full, f"noisy_input_{image_index}.tiff")),
        "Noisy prediction": _read(os.path.join(full, f"noisy_unet_prediction_{image_index}.tiff")),
    }
    ref = imgs["Prediction"]
    if ref is None:
        print("no prediction tiff at", base)
        return
    z = ref.shape[0] // 2 if z_slice is None else z_slice

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, (title, vol) in zip(axes.ravel(), imgs.items()):
        ax.set_title(title)
        ax.axis("off")
        if vol is None:
            continue
        cmap = "hot" if title == "Importance mask" else "gray"
        data = vol[z] if title == "Importance mask" else auto_balance(vol[z])
        im = ax.imshow(data, cmap=cmap, vmin=0 if title == "Importance mask" else None,
                       vmax=1 if title == "Importance mask" else None)
        if title == "Importance mask":
            fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle(f"FOV {image_index} — z-slice {z}/{ref.shape[0]}", fontweight="bold")
    fig.tight_layout()
    plt.show()


visualize_results(OUT, image_index=0)

## Next steps

- **Real data**: build `FOVS` from your own `(Z, Y, X)` arrays (or pass a channel-first
  `(C, Z, Y, X)` array per FOV with integer `input_col`/`target_col`), and set
  `PATCH`/`xy_step`/`z_step` to your acquisition size (`(32, 128, 128, 1)` is the paper
  default). To stream from disk instead, use `FOVPatchDataset(image_list_csv=...)` /
  `Analyzer(data=...)` with a CSV of `path_tiff` + channel-index columns.
- **2D**: use `(Y, X)` volumes with `patch_size=(1, Y, X)` (dataset) / `(1, Y, X, 1)`
  (analyzer) and build the interpreter with `ndim=2`.
- **Real predictor**: `freeze()` your pretrained in-silico UNet and pass it to
  `Image2ImageInterpreter`; load a trained mask generator with
  `interp.load_state_dict(torch.load("mask_generator.pt"))`.
- **Other tasks**: `ClassificationInterpreter` and `RegressionInterpreter` interpret
  vector-output predictors (2D); see `examples/quickstart.py` and the README.
- **Weighted PCC**: pass `weighted_pcc=True` and a segmentation channel to score efficacy
  only within the organelle.